In [32]:
import os
os.chdir(r'C:\wajeeh\University\8th Semester\Machine Learning\ML Lab\Code')

In [33]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, explained_variance_score, r2_score
from timeseires.utils.to_split import to_split
from timeseires.utils.multivariate_multi_step import multivariate_multi_step
from timeseires.utils.multivariate_single_step import multivariate_single_step
from timeseires.utils.univariate_multi_step import univariate_multi_step
from timeseires.utils.univariate_single_step import univariate_single_step
from timeseires.utils.CosineAnnealingLRS import CosineAnnealingLRS
from timeseires.callbacks.EpochCheckpoint import EpochCheckpoint
from tensorflow.keras.callbacks import ModelCheckpoint
from timeseires.callbacks.TrainingMonitor import TrainingMonitor
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.optimizers import SGD
from tensorflow.keras.models import load_model
from tensorflow.keras.layers import LSTM, Bidirectional, Add
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.layers import Conv1D,TimeDistributed
from tensorflow.keras.layers import Dense, Dropout, Activation, Flatten,MaxPooling1D,Concatenate,AveragePooling1D, GlobalMaxPooling1D, Input
from tensorflow.keras.models import Sequential,Model
import pandas as pd
import time, pickle
import numpy as np
import tensorflow.keras.backend as K
import tensorflow
from tensorflow.keras.layers import Input, Reshape, Lambda
from tensorflow.keras.layers import Layer, Flatten, LeakyReLU, concatenate, Dense
from tensorflow.keras.regularizers import l2
import glob
import h5py
import matplotlib.pyplot as plt
from keras.callbacks import Callback

In [34]:
#lookback = 24
model = None
start_epoch = 0
time_steps=24
num_features=21

In [35]:
def CNN():
    input_data = Input(shape=(time_steps, num_features))
    x1 = Conv1D(16, 2, activation="relu")(input_data)
    x2 = Conv1D(16, 2, activation="relu")(x1)
    flatten = Flatten()(x2)
    output_data = Dense(1)(flatten)
    model = Model(input_data, output_data)
    return model

In [36]:
model1 = CNN()
model1.summary()

Model: "model_2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_3 (InputLayer)        [(None, 24, 21)]          0         
                                                                 
 conv1d_4 (Conv1D)           (None, 23, 16)            688       
                                                                 
 conv1d_5 (Conv1D)           (None, 22, 16)            528       
                                                                 
 flatten_2 (Flatten)         (None, 352)               0         
                                                                 
 dense_2 (Dense)             (None, 1)                 353       
                                                                 
Total params: 1569 (6.13 KB)
Trainable params: 1569 (6.13 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [37]:
tensorflow.keras.utils.plot_model(model1 )

You must install pydot (`pip install pydot`) and install graphviz (see instructions at https://graphviz.gitlab.io/download/) for plot_model to work.


In [38]:
checkpoints = r'C:\wajeeh\University\8th Semester\Machine Learning\ML Lab\Code\E1-cp-{epoch:04d}-loss{val_loss:.2f}.h5'
OUTPUT_PATH = r'C:\wajeeh\University\8th Semester\Machine Learning\ML Lab\Code'
FIG_PATH = os.path.sep.join([OUTPUT_PATH,"\history.png"])
JSON_PATH = os.path.sep.join([OUTPUT_PATH,"\history.json"])

In [39]:
# construct the callback to save only the *best* model to disk
# based on the validation loss
EpochCheckpoint1 = ModelCheckpoint(checkpoints,
                             monitor="val_loss",
                             save_best_only=True, 
                             verbose=1)
TrainingMonitor1=TrainingMonitor(FIG_PATH, jsonPath=JSON_PATH, startAt=start_epoch)

# construct the set of callbacks
callbacks = [EpochCheckpoint1,TrainingMonitor1]

In [40]:
# if there is no specific model checkpoint supplied, then initialize
# the network and compile the model
if model is None:
    print("[INFO] compiling model...")
    model =CNN()
    opt = Adam(1e-3)
    model.compile(loss= 'mae', optimizer=opt, metrics=["mae", "mape"])
# otherwise, load the checkpoint from disk
else:
    print("[INFO] loading {}...".format(model))
    model = load_model(model)

    # update the learning rate
    print("[INFO] old learning rate: {}".format(K.get_value(model.optimizer.lr)))
    K.set_value(model.optimizer.lr, 1e-4)
    print("[INFO] new learning rate: {}".format(K.get_value(model.optimizer.lr)))

[INFO] compiling model...


In [41]:
import os
path_dataset =r'C:\wajeeh\University\8th Semester\Machine Learning\ML Lab\Code'
path_tr = os.path.join(path_dataset, 'train.csv')
df_tr = pd.read_csv(path_tr)
train_set = df_tr.iloc[:].values
path_v = os.path.join(path_dataset, 'validation.csv')
df_v = pd.read_csv(path_v)
validation_set = df_v.iloc[:].values 
path_te = os.path.join(path_dataset, 'test.csv')
df_te = pd.read_csv(path_te)
test_set = df_te.iloc[:].values 

path_scaler = os.path.join(path_dataset, 'AEP_scaler.pkl')
scaler         = pickle.load(open(path_scaler, 'rb'))

train_set.shape, validation_set.shape, test_set.shape

c:\Users\MASTER\anaconda3\envs\ml\lib\site-packages\sklearn\base.py:347: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.0.2 when using version 1.3.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


((860, 21), (90, 21), (30, 21))

In [42]:
time_steps=24
num_features=21

In [43]:
start = time.time()
train_X , train_y = univariate_multi_step(train_set, time_steps, target_col=0,target_len=1)
validation_X, validation_y = univariate_multi_step(validation_set, time_steps, target_col=0,target_len=1)
test_X, test_y = univariate_multi_step(test_set, time_steps, target_col=0,target_len=1)
print('Time Consumed', time.time()-start, "sec")

Time Consumed 0.011175155639648438 sec


In [44]:
epochs = 10
verbose = 1 #0
batch_size = 32
History = model.fit(train_X,
                        train_y,
                        batch_size=batch_size,   
                        epochs = epochs, 
                        validation_data = (validation_X,validation_y),
                        callbacks=callbacks,verbose = verbose)

Epoch 1/10
14/27 [==============>...............] - ETA: 0s - loss: 0.2250 - mae: 0.2250 - mape: 101.6896 
Epoch 1: val_loss improved from inf to 0.08340, saving model to C:\wajeeh\University\8th Semester\Machine Learning\ML Lab\Code\E1-cp-0001-loss0.08.h5


c:\Users\MASTER\anaconda3\envs\ml\lib\site-packages\keras\src\engine\training.py:3000: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


27/27 [==============================] - 2s 27ms/step - loss: 0.1722 - mae: 0.1722 - mape: 81.5880 - val_loss: 0.0834 - val_mae: 0.0834 - val_mape: 26.2709
Epoch 2/10
26/27 [===========================>..] - ETA: 0s - loss: 0.0748 - mae: 0.0748 - mape: 35.7031
Epoch 2: val_loss improved from 0.08340 to 0.05777, saving model to C:\wajeeh\University\8th Semester\Machine Learning\ML Lab\Code\E1-cp-0002-loss0.06.h5


c:\Users\MASTER\anaconda3\envs\ml\lib\site-packages\keras\src\engine\training.py:3000: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


27/27 [==============================] - 0s 19ms/step - loss: 0.0748 - mae: 0.0748 - mape: 35.6725 - val_loss: 0.0578 - val_mae: 0.0578 - val_mape: 20.9107
Epoch 3/10
25/27 [==========================>...] - ETA: 0s - loss: 0.0659 - mae: 0.0659 - mape: 35.6330
Epoch 3: val_loss did not improve from 0.05777
27/27 [==============================] - 0s 17ms/step - loss: 0.0654 - mae: 0.0654 - mape: 35.2083 - val_loss: 0.0720 - val_mae: 0.0720 - val_mape: 26.2290
Epoch 4/10
16/27 [================>.............] - ETA: 0s - loss: 0.0646 - mae: 0.0646 - mape: 29.2896
Epoch 4: val_loss improved from 0.05777 to 0.03967, saving model to C:\wajeeh\University\8th Semester\Machine Learning\ML Lab\Code\E1-cp-0004-loss0.04.h5


...

27/27 [==============================] - 0s 17ms/step - loss: 0.0617 - mae: 0.0617 - mape: 26.9000 - val_loss: 0.0397 - val_mae: 0.0397 - val_mape: 13.1456
Epoch 5/10
27/27 [==============================] - ETA: 0s - loss: 0.0518 - mae: 0.0518 - mape: 26.6439
Epoch 5: val_loss did not improve from 0.03967
27/27 [==============================] - 0s 18ms/step - loss: 0.0518 - mae: 0.0518 - mape: 26.6439 - val_loss: 0.0469 - val_mae: 0.0469 - val_mape: 16.8978
Epoch 6/10
16/27 [================>.............] - ETA: 0s - loss: 0.0506 - mae: 0.0506 - mape: 23.5626
Epoch 6: val_loss did not improve from 0.03967
27/27 [==============================] - 0s 17ms/step - loss: 0.0498 - mae: 0.0498 - mape: 24.0515 - val_loss: 0.0641 - val_mae: 0.0641 - val_mape: 22.5791
Epoch 7/10
13/27 [=============>................] - ETA: 0s - loss: 0.0516 - mae: 0.0516 - mape: 20.5482
Epoch 7: val_loss did not improve from 0.03967
27/27 [==============================] - 0s 16ms/step - loss: 0.0502 - mae: 

c:\Users\MASTER\anaconda3\envs\ml\lib\site-packages\keras\src\engine\training.py:3000: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


27/27 [==============================] - 1s 20ms/step - loss: 0.0454 - mae: 0.0454 - mape: 20.9128 - val_loss: 0.0382 - val_mae: 0.0382 - val_mape: 13.2857
Epoch 9/10
12/27 [============>.................] - ETA: 0s - loss: 0.0489 - mae: 0.0489 - mape: 23.0405
Epoch 9: val_loss did not improve from 0.03820
27/27 [==============================] - 0s 17ms/step - loss: 0.0452 - mae: 0.0452 - mape: 21.0037 - val_loss: 0.0463 - val_mae: 0.0463 - val_mape: 16.0387
Epoch 10/10
16/27 [================>.............] - ETA: 0s - loss: 0.0434 - mae: 0.0434 - mape: 20.8430
Epoch 10: val_loss improved from 0.03820 to 0.03692, saving model to C:\wajeeh\University\8th Semester\Machine Learning\ML Lab\Code\E1-cp-0010-loss0.04.h5


c:\Users\MASTER\anaconda3\envs\ml\lib\site-packages\keras\src\engine\training.py:3000: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


27/27 [==============================] - 1s 19ms/step - loss: 0.0411 - mae: 0.0411 - mape: 19.5720 - val_loss: 0.0369 - val_mae: 0.0369 - val_mape: 12.9541


In [45]:

model = load_model(r'C:\wajeeh\University\8th Semester\Machine Learning\ML Lab\Code\E1-cp-0006-loss0.03.h5')

y_pred_scaled   = model.predict(test_X)
y_pred          = scaler.inverse_transform(y_pred_scaled)
y_test_unscaled = scaler.inverse_transform(test_y)
# Mean Absolute Error (MAE)
MAE = np.mean(abs(y_pred - y_test_unscaled)) 
print('Mean Absolute Error (MAE): ' + str(np.round(MAE, 2)))

# Median Absolute Error (MedAE)
MEDAE = np.median(abs(y_pred - y_test_unscaled))
print('Median Absolute Error (MedAE): ' + str(np.round(MEDAE, 2)))

# Mean Squared Error (MSE)
MSE = np.square(np.subtract(y_pred, y_test_unscaled)).mean()
print('Mean Squared Error (MSE): ' + str(np.round(MSE, 2)))

# Root Mean Squarred Error (RMSE) 
RMSE = np.sqrt(np.mean(np.square(y_pred - y_test_unscaled)))
print('Root Mean Squared Error (RMSE): ' + str(np.round(RMSE, 2)))

# Mean Absolute Percentage Error (MAPE)
MAPE = np.mean((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Mean Absolute Percentage Error (MAPE): ' + str(np.round(MAPE, 2)) + ' %')

# Median Absolute Percentage Error (MDAPE)
MDAPE = np.median((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Median Absolute Percentage Error (MDAPE): ' + str(np.round(MDAPE, 2)) + ' %')

print('\n\ny_test_unscaled.shape= ',y_test_unscaled.shape)
print('y_pred.shape= ',y_pred.shape)

1/1 [==============================] - 0s 119ms/step
Mean Absolute Error (MAE): 5142.5
Median Absolute Error (MedAE): 5102.34
Mean Squared Error (MSE): 26600759.38
Root Mean Squared Error (RMSE): 5157.59
Mean Absolute Percentage Error (MAPE): 32.88 %
Median Absolute Percentage Error (MDAPE): 32.97 %


y_test_unscaled.shape=  (5, 1)
y_pred.shape=  (5, 1)


In [46]:
checkpoints = r'C:\wajeeh\University\8th Semester\Machine Learning\ML Lab\Code\E1-cp-0006-loss0.03.h5'
model=r'C:\wajeeh\University\8th Semester\Machine Learning\ML Lab\Code\E1-cp-0006-loss0.03.h5'
start_epoch= 58

In [47]:
# construct the callback to save only the *best* model to disk
# based on the validation loss
EpochCheckpoint1 = ModelCheckpoint(checkpoints,
                             monitor="val_loss",
                             save_best_only=True, 
                             verbose=1)
TrainingMonitor1=TrainingMonitor(FIG_PATH, jsonPath=JSON_PATH, startAt=start_epoch)

# construct the set of callbacks
callbacks = [EpochCheckpoint1,TrainingMonitor1]
# if there is no specific model checkpoint supplied, then initialize
# the network and compile the model
if model is None:
    print("[INFO] compiling model...")
    model = PC.build(time_steps=24, num_features=21, reg=0.0005)
    opt = Adam(1e-3)
    model.compile(loss= 'mae', optimizer=opt, metrics=["mae", "mape"])
# otherwise, load the checkpoint from disk
else:
    print("[INFO] loading {}...".format(model))
    model = load_model(model)

    # update the learning rate
    print("[INFO] old learning rate: {}".format(K.get_value(model.optimizer.lr)))
    K.set_value(model.optimizer.lr, 1e-4)
    print("[INFO] new learning rate: {}".format(K.get_value(model.optimizer.lr)))

[INFO] loading C:\wajeeh\University\8th Semester\Machine Learning\ML Lab\Code\E1-cp-0006-loss0.03.h5...
[INFO] old learning rate: 9.999999747378752e-05
[INFO] new learning rate: 9.999999747378752e-05


In [48]:
epochs = 10
verbose = 1 #0
batch_size = 32
History = model.fit(train_X,
                        train_y,
                        batch_size=batch_size,   
                        epochs = epochs, 
                        validation_data = (validation_X,validation_y),
                        callbacks=callbacks,
                        verbose = verbose)

Epoch 1/10
27/27 [==============================] - ETA: 0s - loss: 0.0099 - mae: 0.0099 - mape: 4.4684
Epoch 1: val_loss improved from inf to 0.03107, saving model to C:\wajeeh\University\8th Semester\Machine Learning\ML Lab\Code\E1-cp-0006-loss0.03.h5


c:\Users\MASTER\anaconda3\envs\ml\lib\site-packages\keras\src\engine\training.py:3000: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


27/27 [==============================] - 1s 29ms/step - loss: 0.0099 - mae: 0.0099 - mape: 4.4684 - val_loss: 0.0311 - val_mae: 0.0311 - val_mape: 9.7081
Epoch 2/10
25/27 [==========================>...] - ETA: 0s - loss: 0.0103 - mae: 0.0103 - mape: 4.6963
Epoch 2: val_loss improved from 0.03107 to 0.03100, saving model to C:\wajeeh\University\8th Semester\Machine Learning\ML Lab\Code\E1-cp-0006-loss0.03.h5


c:\Users\MASTER\anaconda3\envs\ml\lib\site-packages\keras\src\engine\training.py:3000: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


27/27 [==============================] - 0s 19ms/step - loss: 0.0102 - mae: 0.0102 - mape: 4.6519 - val_loss: 0.0310 - val_mae: 0.0310 - val_mape: 9.3346
Epoch 3/10
26/27 [===========================>..] - ETA: 0s - loss: 0.0101 - mae: 0.0101 - mape: 4.4383
Epoch 3: val_loss improved from 0.03100 to 0.03021, saving model to C:\wajeeh\University\8th Semester\Machine Learning\ML Lab\Code\E1-cp-0006-loss0.03.h5


c:\Users\MASTER\anaconda3\envs\ml\lib\site-packages\keras\src\engine\training.py:3000: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


27/27 [==============================] - 1s 19ms/step - loss: 0.0101 - mae: 0.0101 - mape: 4.4312 - val_loss: 0.0302 - val_mae: 0.0302 - val_mape: 9.4472
Epoch 4/10
13/27 [=============>................] - ETA: 0s - loss: 0.0102 - mae: 0.0102 - mape: 3.8392
Epoch 4: val_loss did not improve from 0.03021
27/27 [==============================] - 0s 17ms/step - loss: 0.0099 - mae: 0.0099 - mape: 4.5073 - val_loss: 0.0316 - val_mae: 0.0316 - val_mape: 9.6487
Epoch 5/10
15/27 [===============>..............] - ETA: 0s - loss: 0.0100 - mae: 0.0100 - mape: 4.6640
Epoch 5: val_loss did not improve from 0.03021
27/27 [==============================] - 0s 16ms/step - loss: 0.0098 - mae: 0.0098 - mape: 4.4179 - val_loss: 0.0313 - val_mae: 0.0313 - val_mape: 9.4791
Epoch 6/10
14/27 [==============>...............] - ETA: 0s - loss: 0.0098 - mae: 0.0098 - mape: 4.9452
Epoch 6: val_loss did not improve from 0.03021
27/27 [==============================] - 0s 16ms/step - loss: 0.0098 - mae: 0.0098 - 

In [49]:

model = load_model(r'C:\wajeeh\University\8th Semester\Machine Learning\ML Lab\Code\E1-cp-0006-loss0.03.h5')

y_pred_scaled   = model.predict(test_X)
y_pred          = scaler.inverse_transform(y_pred_scaled)
y_test_unscaled = scaler.inverse_transform(test_y)
# Mean Absolute Error (MAE)
MAE = np.mean(abs(y_pred - y_test_unscaled)) 
print('Mean Absolute Error (MAE): ' + str(np.round(MAE, 2)))

# Median Absolute Error (MedAE)
MEDAE = np.median(abs(y_pred - y_test_unscaled))
print('Median Absolute Error (MedAE): ' + str(np.round(MEDAE, 2)))

# Mean Squared Error (MSE)
MSE = np.square(np.subtract(y_pred, y_test_unscaled)).mean()
print('Mean Squared Error (MSE): ' + str(np.round(MSE, 2)))

# Root Mean Squarred Error (RMSE) 
RMSE = np.sqrt(np.mean(np.square(y_pred - y_test_unscaled)))
print('Root Mean Squared Error (RMSE): ' + str(np.round(RMSE, 2)))

# Mean Absolute Percentage Error (MAPE)
MAPE = np.mean((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Mean Absolute Percentage Error (MAPE): ' + str(np.round(MAPE, 2)) + ' %')

# Median Absolute Percentage Error (MDAPE)
MDAPE = np.median((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Median Absolute Percentage Error (MDAPE): ' + str(np.round(MDAPE, 2)) + ' %')

print('\n\ny_test_unscaled.shape= ',y_test_unscaled.shape)
print('y_pred.shape= ',y_pred.shape)

1/1 [==============================] - 0s 121ms/step
Mean Absolute Error (MAE): 5149.4
Median Absolute Error (MedAE): 5103.28
Mean Squared Error (MSE): 26676929.99
Root Mean Squared Error (RMSE): 5164.97
Mean Absolute Percentage Error (MAPE): 32.93 %
Median Absolute Percentage Error (MDAPE): 32.98 %


y_test_unscaled.shape=  (5, 1)
y_pred.shape=  (5, 1)


# lab report 

## Lab 1